# MinIO + Azure Container Apps + Fabric walkthrough

Use this notebook to provision a test MinIO deployment on Azure Container Apps with Azure Verified Modules and capture the values needed for Microsoft Fabric shortcuts.

## 1) Validate Azure/Bicep tooling

Run the next cell to confirm Azure CLI and Bicep are ready inside Codespaces.

In [ ]:
!az version
!az bicep version

## 2) Review deployment settings

Set or override the deployment values before creating Azure resources. Environment variables let you customize names, location, image, or credentials without editing the notebook.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import secrets
import string
import subprocess
import tempfile

REPO_ROOT = Path('/home/runner/work/fabric-minio/fabric-minio')
TEMPLATE_FILE = REPO_ROOT / 'infra' / 'main.bicep'
TEMP_DIR = Path(tempfile.gettempdir()) / 'fabric-minio'
TEMP_DIR.mkdir(parents=True, exist_ok=True)

def random_suffix(length: int = 5) -> str:
    alphabet = string.ascii_lowercase + string.digits
    return ''.join(secrets.choice(alphabet) for _ in range(length))

def generate_secret(length: int = 24) -> str:
    alphabet = string.ascii_letters + string.digits
    return ''.join(secrets.choice(alphabet) for _ in range(length))

suffix = random_suffix()
timestamp = datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')
deployment_config = {
    'location': os.getenv('AZURE_LOCATION', 'westeurope'),
    'resource_group_name': os.getenv('RESOURCE_GROUP_NAME', f'rg-fabric-minio-{suffix}'),
    'deployment_name': os.getenv('DEPLOYMENT_NAME', f'fabric-minio-{timestamp}'),
    'log_analytics_workspace_name': os.getenv('LOG_ANALYTICS_WORKSPACE_NAME', f'log-fabric-minio-{suffix}'),
    'container_apps_environment_name': os.getenv('CONTAINER_APPS_ENVIRONMENT_NAME', f'cae-fabric-minio-{suffix}'),
    'container_app_name': os.getenv('CONTAINER_APP_NAME', f'minio-fabric-{suffix}'),
    'minio_image': os.getenv('MINIO_IMAGE', 'quay.io/minio/minio:latest'),
    'minio_root_user': os.getenv('MINIO_ROOT_USER', 'fabricminio'),
    'minio_root_password': os.getenv('MINIO_ROOT_PASSWORD', generate_secret()),
    'tags': {
        'workload': 'fabric-minio',
        'purpose': 'testing'
    }
}
deployment_config

## 3) Verify Azure login

Sign in with `az login` in a terminal first if this cell fails.

In [ ]:
!az account show --output table

## 4) Create the resource group

The notebook keeps resource group creation separate from the AVM deployment so you can reuse or delete the group independently.

In [ ]:
create_rg = [
    'az', 'group', 'create',
    '--name', deployment_config['resource_group_name'],
    '--location', deployment_config['location'],
    '--output', 'json'
]
print(' '.join(create_rg))
resource_group = json.loads(subprocess.run(create_rg, check=True, capture_output=True, text=True).stdout)
resource_group['id']

## 5) Validate the Bicep template

This catches local template issues before Azure deployment starts.

In [ ]:
!az bicep build --file /home/runner/work/fabric-minio/fabric-minio/infra/main.bicep

## 6) Deploy the Azure Verified Modules template

The next cell writes secure deployment parameters to `/tmp`, runs the group deployment, and captures the template outputs.

In [ ]:
parameters_file = TEMP_DIR / 'main.parameters.json'
parameters_payload = {
    '$schema': 'https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#',
    'contentVersion': '1.0.0.0',
    'parameters': {
        'location': {'value': deployment_config['location']},
        'logAnalyticsWorkspaceName': {'value': deployment_config['log_analytics_workspace_name']},
        'containerAppsEnvironmentName': {'value': deployment_config['container_apps_environment_name']},
        'containerAppName': {'value': deployment_config['container_app_name']},
        'minioImage': {'value': deployment_config['minio_image']},
        'minioRootUser': {'value': deployment_config['minio_root_user']},
        'minioRootPassword': {'value': deployment_config['minio_root_password']},
        'tags': {'value': deployment_config['tags']}
    }
}
parameters_file.write_text(json.dumps(parameters_payload, indent=2), encoding='utf-8')
deployment_cmd = [
    'az', 'deployment', 'group', 'create',
    '--name', deployment_config['deployment_name'],
    '--resource-group', deployment_config['resource_group_name'],
    '--template-file', str(TEMPLATE_FILE),
    '--parameters', f'@{parameters_file}',
    '--query', 'properties.outputs',
    '--output', 'json'
]
print(' '.join(deployment_cmd[:-1]))
deployment_outputs = json.loads(subprocess.run(deployment_cmd, check=True, capture_output=True, text=True).stdout)
deployment_outputs

## 7) Copy the values into Microsoft Fabric

Use the printed endpoint as the S3-compatible service URL for Fabric shortcuts and the generated access key pair as the connection credentials.

In [ ]:
connection_info = {
    'fabricShortcutServiceUrl': deployment_outputs['fabricShortcutServiceUrl']['value'],
    'fabricShortcutHost': deployment_outputs['fabricShortcutHost']['value'],
    'accessKeyId': deployment_config['minio_root_user'],
    'secretAccessKey': deployment_config['minio_root_password'],
    'containerAppResourceId': deployment_outputs['containerAppResourceId']['value']
}
connection_info